# Train the thermal cascade — YOLO26x (transformer + wire)
Runtime → Change runtime type → **GPU** first.

Trains **two single-class detectors** on the datasets built by `thermal.data_prep.build` (already deduped + CLAHE-preprocessed):
- `transformer` — full frame
- `wire` — on transformer crops (the cascade)

Both train on the `clahe` image variant, with thermal-tuned augmentation (**no hue jitter** — it would scramble the heat palette).

In [ ]:
# YOLO26 gotchas (learned on Colab): 8.3.x silently degrades yolo26 -> nano,
# and pillow 11.3+ is broken on Colab. Pin both.
!pip install -q -U "ultralytics>=8.4.60" "pillow==11.2.1"

### Upload the datasets
On your machine, zip each dataset's **clahe** variant (smaller upload):
```bash
cd /Volumes/dronisight
zip -r transformer.zip yolo_thermal_transformer/data_clahe.yaml \
    yolo_thermal_transformer/images/*/clahe yolo_thermal_transformer/labels/*/clahe
zip -r wire.zip Yolo_thermal_wire/data_clahe.yaml \
    Yolo_thermal_wire/images/*/clahe Yolo_thermal_wire/labels/*/clahe
```
Then run the next cell and pick **both** zips.

In [ ]:
import zipfile, os
from google.colab import files
for name, up in files.upload().items():
    with zipfile.ZipFile(name) as z:
        z.extractall('/content')
    print('extracted', name)
print(os.listdir('/content'))

In [ ]:
# The build-time data.yaml `path:` points at the build machine; repoint to /content.
import yaml, glob
DATASETS = {
    'transformer': '/content/yolo_thermal_transformer',
    'wire': '/content/Yolo_thermal_wire',
}
DATA_YAML = {}
for key, root in DATASETS.items():
    p = f'{root}/data_clahe.yaml'
    d = yaml.safe_load(open(p)); d['path'] = root
    yaml.safe_dump(d, open(p, 'w'), sort_keys=False)
    DATA_YAML[key] = p
    print(key, '->', d)

In [ ]:
# Thermal-tuned training. NO hue jitter (palette = heat). x is heavy for ~600-700
# train imgs -> early stopping + regularization. Drop to yolo26m.pt / lower batch on OOM.
MODEL = 'yolo26x.pt'
def train_args(data_yaml, scale):
    return dict(
        data=data_yaml, epochs=150, imgsz=1280, batch=4, seed=1337,
        hsv_h=0.0, hsv_s=0.2, hsv_v=0.3,
        fliplr=0.5, flipud=0.0, degrees=10.0, translate=0.1, scale=scale,
        mosaic=1.0, close_mosaic=10,
        weight_decay=0.0005, dropout=0.1, cos_lr=True, patience=30, amp=True,
    )

In [ ]:
from ultralytics import YOLO
# full frame, large object -> moderate scale jitter
m_t = YOLO(MODEL)
m_t.train(project='runs/transformer', name='yolo26x', **train_args(DATA_YAML['transformer'], scale=0.5))

In [ ]:
# transformer crops, thin objects -> wider scale jitter to mimic crop zoom
m_w = YOLO(MODEL)
m_w.train(project='runs/wire', name='yolo26x', **train_args(DATA_YAML['wire'], scale=0.9))

In [ ]:
for tag, m in (('transformer', m_t), ('wire', m_w)):
    r = m.val()
    print(f'{tag}: mAP50-95={r.box.map:.3f}  mAP50={r.box.map50:.3f}')

In [ ]:
# Download both best.pt, renamed for the cascade.
import shutil
from google.colab import files
shutil.copy('runs/transformer/yolo26x/weights/best.pt', 'transformer.pt')
shutil.copy('runs/wire/yolo26x/weights/best.pt', 'wire.pt')
files.download('transformer.pt'); files.download('wire.pt')

Put `transformer.pt` and `wire.pt` into the repo's `models/` folder, then run the cascade inference / API. Watch `results.png` per run: a widening train-vs-val gap = overfitting → try `MODEL='yolo26m.pt'` or fewer epochs.